In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import torch
import numpy as np
import yfinance as yf
from models.lstm_model import LSTM_Model
from utils import get_device

# Parameter

In [10]:
ckpt_path = '../checkpoints/cnn_checkpoint.pt'

ticker = 'VOO'
predict_from = '2026-01-01'
predict_to   = '2026-03-31'

# Extract saved objects and configs

In [4]:
device = get_device()
checkpoint = torch.load("../checkpoints/lstm_checkpoint.pt", map_location=device)

In [5]:
model_config = checkpoint["model_config"]
x_scaler = checkpoint["x_scaler"]
y_scaler = checkpoint["y_scaler"]
input_cols = checkpoint["input_cols"]
seq_len = checkpoint["preprocess_config"]["seq_length"]

# Rebuild model

In [8]:
model = LSTM_Model(**model_config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
_ = model.eval()

# Get Latest Data

In [11]:
df = yf.download(ticker, period="6mo", auto_adjust=False)
data = df.dropna().copy()

[*********************100%***********************]  1 of 1 completed


In [12]:
X_latest = x_scaler.transform(data[input_cols].values.astype(np.float32))

(124, 5)